In [ ]:
!pip install -q pandas numpy scikit-learn nltk tqdm rank_bm25 networkx matplotlib \
    langchain-text-splitters sentence-transformers bertopic umap-learn hdbscan optuna groq gradio

In [ ]:
import os, re, math, time, getpass, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Google Colab: mount Drive (skipped automatically elsewhere)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

DATA_PATH = "/content/ipc_sections.csv"      # change if the CSV lives elsewhere (e.g. in Drive)
SEED = 42
np.random.seed(SEED)

# ---- Fixed settings from the paper (Table 2) ----
CHUNK_SIZE, CHUNK_OVERLAP, MIN_CHUNK_LEN = 700, 100, 100
BM25_K1, BM25_B = 1.5, 0.75
RRF_KAPPA = 60
K_FINAL = 5                 # final context size (not tuned)
GRAPH_THETA = 0.85          # cosine threshold for similarity edges
TOPIC_GAMMA = 0.5           # down-weight for chunks outside the query's topic (Eq. 7)
N_TRIALS = 30               # Optuna trials

# Groq key: taken from Colab Secrets (key icon -> GROQ_API_KEY) or typed when asked. Nothing is stored in the notebook.
if not os.environ.get("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Raw records:", len(df), "| columns:", df.columns.tolist())

# Deduplication and empty-record removal
df = df.dropna(subset=["Section", "Description"]).drop_duplicates().reset_index(drop=True)
df["Offense"] = df["Offense"].fillna("Not Specified")
df["Punishment"] = df["Punishment"].fillna("Not Specified")
print("Unique statutory documents:", len(df))

In [ ]:
# Boilerplate removal: three template phrases that occur in almost every record
BOILERPLATE = [
    r"Description of IPC Section\s+\w+",            # heading
    r"According to section\s+\w+\s+of Indian penal code,?", # lead-in
    r"IPC\s+\w+\s+in Simple Words",                  # heading before the plain-language text
]

def clean_text(t):
    t = str(t)
    for pat in BOILERPLATE:
        t = re.sub(pat, " ", t, flags=re.I)
    t = t.replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
    t = re.sub(r"[ \t]+", " ", t)              # extra spaces
    t = re.sub(r"\s*\n\s*", "\n", t)           # blank-line runs
    t = re.sub(r"\s+([,.;:])", r"\1", t)       # space before punctuation
    return t.strip()

for col in ["Description", "Offense", "Punishment"]:
    df[col] = df[col].apply(clean_text)

# Field concatenation -> combined_text
df["combined_text"] = (
    "Section: " + df["Section"].astype(str) +
    "\nOffense: " + df["Offense"] +
    "\nDescription: " + df["Description"] +
    "\nPunishment: " + df["Punishment"]
)
df["Section"] = df["Section"].astype(str)
print(df["combined_text"].iloc[0])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""],
)

rows = []
for _, r in df.iterrows():
    pieces = [p.strip() for p in splitter.split_text(r["combined_text"])]
    kept = [p for p in pieces if len(p) >= MIN_CHUNK_LEN]
    if not kept and pieces:            # never lose a whole short section
        kept = [max(pieces, key=len)]
    for i, p in enumerate(kept, 1):
        rows.append({"Section": r["Section"], "Chunk_ID": f"{r['Section']}_{i}", "Chunk": p})

chunk_df = pd.DataFrame(rows)
print("Total chunks:", len(chunk_df), "| sections covered:", chunk_df["Section"].nunique(), "/", len(df))
chunk_df.head()

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("BAAI/bge-m3", device=device)

embedding_matrix = embedding_model.encode(
    chunk_df["Chunk"].tolist(),
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,        # unit length -> cosine == dot product (Eq. 4)
)
print("Embedding matrix:", embedding_matrix.shape)

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

topic_model = BERTopic(
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED),
    hdbscan_model=HDBSCAN(min_cluster_size=10, metric="euclidean", prediction_data=True),
    calculate_probabilities=False,
    verbose=False,
)
topics, _ = topic_model.fit_transform(chunk_df["Chunk"].tolist(), embedding_matrix)
chunk_df["Topic"] = topics
print("Topics found (excluding noise -1):", len(set(topics) - {-1}))

# Topic centroids in embedding space: tau(q) = nearest centroid (noise cluster excluded)
topic_ids = sorted(t for t in set(topics) if t != -1)
topic_centroids = np.vstack([embedding_matrix[chunk_df["Topic"].values == t].mean(axis=0) for t in topic_ids])
topic_centroids /= np.linalg.norm(topic_centroids, axis=1, keepdims=True)

topic_model.get_topic_info().head(15)

In [ ]:
import networkx as nx
from itertools import combinations

G = nx.Graph()
for i, r in chunk_df.iterrows():
    G.add_node(i, chunk_id=r["Chunk_ID"], section=r["Section"], topic=int(r["Topic"]))

def add_edge(i, j, w, rel):
    if i == j:
        return
    if G.has_edge(i, j):
        if w > G[i][j]["weight"]:
            G[i][j]["weight"] = w
    else:
        G.add_edge(i, j, weight=w, relation=rel)

# (a) shared topic  (noise topic -1 is not a real topic, so it creates no edges)
W_TOPIC = 0.5
for t in topic_ids:
    members = chunk_df.index[chunk_df["Topic"] == t].tolist()
    for i, j in combinations(members, 2):
        add_edge(i, j, W_TOPIC, "topic")

# (b) explicit cross-reference, e.g. "Section 302 refers to Section 300"
W_XREF = 1.0
sec_to_idx = chunk_df.groupby("Section").groups
xref_pat = re.compile(r"[Ss]ections?\s+(\d+[A-Z]?)")
n_xref = 0
for i, r in chunk_df.iterrows():
    own = r["Section"].replace("IPC_", "")
    for ref in set(xref_pat.findall(r["Chunk"])):
        if ref != own and f"IPC_{ref}" in sec_to_idx:
            for j in sec_to_idx[f"IPC_{ref}"]:
                add_edge(i, j, W_XREF, "xref"); n_xref += 1

# (c) cosine similarity above theta
sim = embedding_matrix @ embedding_matrix.T
ii, jj = np.where(np.triu(sim >= GRAPH_THETA, k=1))
for i, j in zip(ii, jj):
    add_edge(int(i), int(j), float(sim[i, j]), "similarity")
del sim

print(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()} | cross-reference links: {n_xref}")

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ---- BM25 index (built once) ----
def tok(text):
    return re.findall(r"[a-z0-9]+", text.lower())

bm25 = BM25Okapi([tok(c) for c in chunk_df["Chunk"]], k1=BM25_K1, b=BM25_B)

# ---- Cross-Encoder (MiniLM) with a cache so Optuna does not re-score the same pair twice ----
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)
_ce_cache = {}
def ce_scores(query, idxs):
    need = [i for i in idxs if (query, i) not in _ce_cache]
    if need:
        out = reranker.predict([(query, chunk_df["Chunk"].iat[i]) for i in need], batch_size=32, show_progress_bar=False)
        for i, s in zip(need, out):
            _ce_cache[(query, i)] = float(s)
    return np.array([_ce_cache[(query, i)] for i in idxs])

sigmoid = lambda x: 1.0 / (1.0 + np.exp(-x))
minmax = lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else np.zeros_like(x)

# ---- Per-query first-stage signals (cached: they do not depend on the tuned parameters) ----
_signal_cache = {}
def first_stage(query, per_channel=50):
    if query in _signal_cache:
        return _signal_cache[query]
    q_emb = embedding_model.encode(query, normalize_embeddings=True)
    sem = embedding_matrix @ q_emb                          # Eq. 4
    bm = np.asarray(bm25.get_scores(tok(query)))            # Eq. 5
    # topic of the query = nearest centroid;  TopicScore = 1 or gamma  (Eq. 7)
    q_topic = topic_ids[int(np.argmax(topic_centroids @ q_emb))]
    topic_score = np.where(chunk_df["Topic"].values == q_topic, 1.0, TOPIC_GAMMA)

    def ranked(scores):
        return list(np.argsort(-scores)[:per_channel])

    sem_rank, bm_rank = ranked(sem), ranked(bm)
    topic_rank = ranked(sem * topic_score)                  # topic-filtered semantic list

    # seed set R0 from the three non-graph channels, then one-hop expansion (Eq. 8)
    seed_rrf = {}
    for lst in (sem_rank, bm_rank, topic_rank):
        for r, c in enumerate(lst[:10], 1):
            seed_rrf[c] = seed_rrf.get(c, 0.0) + 1.0 / (RRF_KAPPA + r)
    graph_score = {}
    for c_i, s_i in seed_rrf.items():
        nbrs = G[c_i]
        tot = sum(d["weight"] for d in nbrs.values()) or 1.0
        for c_j, d in nbrs.items():
            graph_score[c_j] = graph_score.get(c_j, 0.0) + (d["weight"] / tot) * s_i
    graph_rank = sorted(graph_score, key=graph_score.get, reverse=True)[:per_channel]

    out = dict(sem=sem, bm=bm, topic_score=topic_score,
               lists=dict(semantic=sem_rank, bm25=bm_rank, topic=topic_rank, graph=graph_rank))
    _signal_cache[query] = out
    return out

def rrf_fuse(lists, kappa=RRF_KAPPA):
    # Eq. 9: a chunk missing from a list contributes nothing for that channel
    score = {}
    for lst in lists:
        for r, c in enumerate(lst, 1):
            score[c] = score.get(c, 0.0) + 1.0 / (kappa + r)
    return sorted(score, key=score.get, reverse=True)

def to_sections(chunk_idx_ordered, k=K_FINAL):
    # evaluation is at section level: keep the best-ranked chunk of each section
    seen, out = set(), []
    for i in chunk_idx_ordered:
        s = chunk_df["Section"].iat[i]
        if s not in seen:
            seen.add(s); out.append(i)
        if len(out) == k:
            break
    return out

def retrieve(query, alpha=0.7784, beta=0.2083, k_candidates=22, k_final=K_FINAL, use_ce=True):
    """Full pipeline. Returns a DataFrame of the top-k_final sections (best chunk each)."""
    sig = first_stage(query)
    pool = rrf_fuse(sig["lists"].values())[:k_candidates]           # high-recall candidate pool
    sem, bm = sig["sem"][pool], sig["bm"][pool]
    hybrid = alpha * sem + (1 - alpha) * minmax(bm)                 # Eq. 10
    if use_ce:
        final = (1 - beta) * hybrid + beta * sigmoid(ce_scores(query, pool))   # Eq. 12
    else:
        final = hybrid
    order = [pool[i] for i in np.argsort(-final)]
    keep = to_sections(order, k_final)
    res = chunk_df.iloc[keep].copy()
    pos = {c: i for i, c in enumerate(pool)}
    res["FinalScore"] = [float(final[pos[c]]) for c in keep]
    return res

print(retrieve("What is the punishment for kidnapping?")[["Section", "Chunk_ID", "FinalScore"]])

In [ ]:
def prf(retrieved, relevant, k=5):
    ret, rel = retrieved[:k], set(relevant)
    tp = len(set(ret) & rel)
    p = tp / len(ret) if ret else 0.0
    r = tp / len(rel) if rel else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

def mrr_at_k(retrieved, relevant, k=5):
    rel = set(relevant)
    for rank, s in enumerate(retrieved[:k], 1):
        if s in rel:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(retrieved, relevant, k=5):
    rel = set(relevant)
    dcg = sum(1 / math.log2(i + 2) for i, s in enumerate(retrieved[:k]) if s in rel)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(len(rel), k)))
    return dcg / idcg if idcg else 0.0

def evaluate(queries, retriever, k=5):
    rows = []
    for q, truth in queries:
        got = retriever(q)["Section"].tolist()
        p, r, f = prf(got, truth, k)
        rows.append({"Query": q, "Retrieved": ", ".join(got), "Ground Truth": ", ".join(truth),
                     "Precision@5": p, "Recall@5": r, "F1@5": f,
                     "MRR": mrr_at_k(got, truth, k), "nDCG@5": ndcg_at_k(got, truth, k),
                     "Hit@5": float(len(set(got[:k]) & set(truth)) > 0)})
    return pd.DataFrame(rows)

def summarise(res):
    return {"Precision@5": res["Precision@5"].mean(), "Recall@5": res["Recall@5"].mean(),
            "F1@5": res["F1@5"].mean(), "MRR": res["MRR"].mean(), "nDCG@5": res["nDCG@5"].mean(),
            "Retrieval Accuracy@5": res["Hit@5"].mean()}

In [ ]:
TEST_QUERIES = [
    ('What is kidnapping?', ['IPC_359', 'IPC_360', 'IPC_361', 'IPC_364A', 'IPC_368']),
    ('What is murder?', ['IPC_299', 'IPC_300', 'IPC_302']),
    ('What is theft?', ['IPC_378', 'IPC_379', 'IPC_380', 'IPC_381', 'IPC_382', 'IPC_390']),
    ('What is cheating?', ['IPC_415', 'IPC_416', 'IPC_417', 'IPC_418']),
    ('What is forgery?', ['IPC_463', 'IPC_464', 'IPC_465', 'IPC_468', 'IPC_469', 'IPC_470']),
    ('What is robbery?', ['IPC_390', 'IPC_392', 'IPC_394']),
    ('What is extortion?', ['IPC_383', 'IPC_384', 'IPC_385']),
    ('What is criminal breach of trust?', ['IPC_405', 'IPC_406', 'IPC_408', 'IPC_409']),
    ('What is defamation?', ['IPC_499', 'IPC_500']),
    ('What is rape?', ['IPC_375', 'IPC_376']),
    ('What is the punishment for dacoity?', ['IPC_391', 'IPC_395', 'IPC_396', 'IPC_397']),
    ('Someone beat me up and broke my bone, which section applies?', ['IPC_320', 'IPC_322', 'IPC_325', 'IPC_326']),
    ('What is the punishment for voluntarily causing hurt?', ['IPC_319', 'IPC_321', 'IPC_323']),
    ('What is the punishment for attempt to murder?', ['IPC_307']),
    ('What is culpable homicide not amounting to murder?', ['IPC_299', 'IPC_304']),
    ('What is dowry death?', ['IPC_304B']),
    ('My husband and in-laws harass me for dowry, which law applies?', ['IPC_498A']),
    ('What is the punishment for abetment of suicide?', ['IPC_305', 'IPC_306']),
    ('Is attempt to commit suicide an offence?', ['IPC_309']),
    ('What is wrongful restraint?', ['IPC_339', 'IPC_341']),
    ('What is wrongful confinement?', ['IPC_340', 'IPC_342', 'IPC_343', 'IPC_344', 'IPC_346']),
    ('Someone threatened to kill me, which section applies?', ['IPC_503', 'IPC_506']),
    ('A man insulted the modesty of a woman by words or gestures', ['IPC_354', 'IPC_509']),
    ('What is stalking?', ['IPC_354D']),
    ('What is voyeurism?', ['IPC_354C']),
    ('What is sexual harassment?', ['IPC_354A']),
    ('Assault on a woman with intent to disrobe her', ['IPC_354B']),
    ('What is the punishment for acid attack?', ['IPC_326A', 'IPC_326B']),
    ('What is human trafficking?', ['IPC_370', 'IPC_370A']),
    ('Buying or selling a person as a slave', ['IPC_370', 'IPC_371']),
    ('What is unlawful compulsory labour?', ['IPC_374']),
    ('Kidnapping a person to demand ransom', ['IPC_364A']),
    ('Taking a minor away from the lawful guardian', ['IPC_361', 'IPC_363']),
    ('Kidnapping or abducting a person in order to murder', ['IPC_364']),
    ('What is abduction?', ['IPC_362', 'IPC_366']),
    ('Kidnapping a woman to force her into marriage', ['IPC_366']),
    ('Inducing a minor girl to go from one place to another for illicit intercourse', ['IPC_366A']),
    ('Selling or hiring a minor for prostitution', ['IPC_372', 'IPC_373']),
    ('What is rioting?', ['IPC_146', 'IPC_147', 'IPC_148']),
    ('What is an unlawful assembly?', ['IPC_141', 'IPC_143', 'IPC_144']),
    ('What is affray?', ['IPC_159', 'IPC_160']),
    ('Promoting enmity between different groups on grounds of religion or race', ['IPC_153A']),
    ('What is sedition?', ['IPC_124A']),
    ('What is waging war against the Government of India?', ['IPC_121', 'IPC_121A']),
    ('Deliberate acts to hurt religious feelings of any class', ['IPC_295A', 'IPC_298']),
    ('Damaging or defiling a place of worship', ['IPC_295']),
    ('Disturbing a religious assembly', ['IPC_296']),
    ('Trespassing on a burial place', ['IPC_297']),
    ('What is criminal trespass?', ['IPC_441', 'IPC_447']),
    ('What is house trespass?', ['IPC_442', 'IPC_448']),
    ('What is housebreaking?', ['IPC_443', 'IPC_445', 'IPC_453', 'IPC_454']),
    ('Breaking into a house at night to commit an offence', ['IPC_456', 'IPC_457']),
    ('What is mischief?', ['IPC_425', 'IPC_426']),
    ('Setting fire to property to cause damage', ['IPC_435', 'IPC_436']),
    ('Burning down a house with intent to destroy it', ['IPC_436']),
    ('Damaging a public road, bridge or water supply', ['IPC_430', 'IPC_431', 'IPC_432']),
    ('What is criminal misappropriation of property?', ['IPC_403', 'IPC_404']),
    ('What is the punishment for receiving stolen property?', ['IPC_410', 'IPC_411', 'IPC_412', 'IPC_414']),
    ('What is cheating by personation?', ['IPC_416', 'IPC_419']),
    ('Dishonestly inducing a person to deliver property by cheating', ['IPC_420']),
    ('Fraudulent transfer of property to defeat creditors', ['IPC_421', 'IPC_422', 'IPC_423', 'IPC_424']),
    ('What is counterfeiting of currency notes?', ['IPC_489A', 'IPC_489B', 'IPC_489C', 'IPC_489D', 'IPC_489E']),
    ('What is counterfeiting a coin?', ['IPC_231', 'IPC_232', 'IPC_233', 'IPC_234']),
    ('What is the punishment for false evidence?', ['IPC_191', 'IPC_193']),
    ('What is perjury in court?', ['IPC_191', 'IPC_192', 'IPC_193']),
    ('What is fabricating false evidence?', ['IPC_192', 'IPC_193']),
    ('Destroying a document or evidence to prevent it being produced', ['IPC_201', 'IPC_204']),
    ('What is the punishment for harbouring an offender?', ['IPC_212', 'IPC_216']),
    ('A public servant disobeying the law to cause injury to a person', ['IPC_166', 'IPC_167']),
    ('A public servant accepting a bribe or illegal gratification', ['IPC_161', 'IPC_162', 'IPC_163']),
    ('Pretending to be a public servant', ['IPC_170', 'IPC_171']),
    ('Obstructing a public servant in discharge of his duty', ['IPC_186', 'IPC_353']),
    ('Assaulting a public servant to deter him from his duty', ['IPC_332', 'IPC_353']),
    ('Disobedience to an order promulgated by a public servant', ['IPC_188']),
    ('Absconding to avoid service of summons', ['IPC_172', 'IPC_174']),
    ('What is bigamy?', ['IPC_494', 'IPC_495']),
    ('What is adultery?', ['IPC_497']),
    ('Enticing or taking away a married woman', ['IPC_498']),
    ('Going through a fraudulent marriage ceremony without lawful marriage', ['IPC_496']),
    ('What are unnatural offences?', ['IPC_377']),
    ('Obscene acts or songs in a public place', ['IPC_294']),
    ('Sale of obscene books or objects', ['IPC_292', 'IPC_293']),
    ('Doing a rash act that endangers human life', ['IPC_336', 'IPC_337', 'IPC_338']),
    ('What is the punishment for rash driving on a public way?', ['IPC_279', 'IPC_304A']),
    ('What is causing death by negligence?', ['IPC_304A']),
    ('Negligent conduct with fire, combustible matter or explosives', ['IPC_285', 'IPC_286', 'IPC_287']),
    ('What is the punishment for adulteration of food or drink?', ['IPC_272', 'IPC_273']),
    ('What is the punishment for adulterating drugs?', ['IPC_274', 'IPC_275', 'IPC_276']),
    ('Negligent act likely to spread infection of a dangerous disease', ['IPC_269', 'IPC_270', 'IPC_271']),
    ('Fouling the water of a public spring or reservoir', ['IPC_277']),
    ('What is a public nuisance?', ['IPC_268', 'IPC_290']),
    ('Using false weights and measures', ['IPC_264', 'IPC_265', 'IPC_266', 'IPC_267']),
    ('Counterfeiting a trade mark or property mark', ['IPC_482', 'IPC_483', 'IPC_486']),
    ('What is criminal conspiracy?', ['IPC_120A', 'IPC_120B']),
    ('What is abetment?', ['IPC_107', 'IPC_108', 'IPC_109']),
    ('What is common intention?', ['IPC_34', 'IPC_35']),
    ('What is the right of private defence of the body?', ['IPC_96', 'IPC_97', 'IPC_100', 'IPC_103']),
    ('What is an attempt to commit an offence?', ['IPC_511']),
    ('What does dishonestly and fraudulently mean in the Indian Penal Code?', ['IPC_23', 'IPC_24', 'IPC_25']),
    ('Printing or engraving defamatory matter', ['IPC_501', 'IPC_502']),
]

In [ ]:
VAL_QUERIES = [
    ('What is the punishment for assault?', ['IPC_351', 'IPC_352', 'IPC_353']),
    ('What is causing grievous hurt by dangerous weapons?', ['IPC_324', 'IPC_326']),
    ('What is the offence of kidnapping a minor boy?', ['IPC_361', 'IPC_363']),
    ('Dishonest misappropriation of the property of a deceased person', ['IPC_404']),
    ('What is criminal force?', ['IPC_349', 'IPC_350', 'IPC_352']),
    ('Robbery with an attempt to cause death or grievous hurt', ['IPC_393', 'IPC_397']),
    ('What is dacoity with murder?', ['IPC_396']),
    ('Making preparation to commit dacoity', ['IPC_399', 'IPC_402']),
    ('Using a forged document as genuine', ['IPC_471']),
    ('Counterfeiting a government stamp or seal', ['IPC_255', 'IPC_256', 'IPC_257', 'IPC_258']),
    ('Theft by a clerk or servant of property in possession of the master', ['IPC_381']),
    ('What is theft in a dwelling house?', ['IPC_380']),
    ('Uttering words with intent to wound religious feelings of a person', ['IPC_298']),
    ('Mischief by killing or maiming an animal', ['IPC_428', 'IPC_429']),
    ('Causing hurt by means of poison', ['IPC_328']),
    ('What is causing miscarriage?', ['IPC_312', 'IPC_313', 'IPC_314', 'IPC_315', 'IPC_316']),
    ('Secret disposal of the body of a child to conceal its birth', ['IPC_318']),
    ('Exposure and abandonment of a child under twelve years', ['IPC_317']),
    ('Rape by a person in custody or by a public servant', ['IPC_376', 'IPC_376C']),
    ('What is the punishment for gang rape?', ['IPC_376D']),
    ('Dishonest breach of trust by a carrier, banker or agent', ['IPC_407', 'IPC_408', 'IPC_409']),
    ('False personation for the purpose of an act or proceeding in a suit', ['IPC_205', 'IPC_209']),
    ('Forging a will or a valuable security', ['IPC_467']),
    ('What is making a false document?', ['IPC_464']),
    ('Possessing counterfeit coin knowing it to be counterfeit', ['IPC_242', 'IPC_243']),
    ('Wrongful confinement to extort a confession', ['IPC_348']),
    ('Causing hurt to extort property', ['IPC_327', 'IPC_329']),
    ('Extortion by threatening to accuse a person of an offence', ['IPC_388', 'IPC_389']),
    ('Putting a person in fear of death in order to commit extortion', ['IPC_386', 'IPC_387']),
    ('Assault or criminal force with intent to dishonour a person', ['IPC_355', 'IPC_356']),
]

In [ ]:
assert len(TEST_QUERIES) == 100, len(TEST_QUERIES)
assert not ({q for q, _ in TEST_QUERIES} & {q for q, _ in VAL_QUERIES}), "validation and test queries overlap"

known = set(df["Section"])
missing = sorted({s for _, secs in TEST_QUERIES + VAL_QUERIES for s in secs if s not in known})
print(f"Test queries: {len(TEST_QUERIES)} | Validation queries: {len(VAL_QUERIES)}")
print("Ground-truth sections missing from the CSV:", missing if missing else "none")

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    alpha = trial.suggest_float("alpha", 0.1, 0.95)
    beta = trial.suggest_float("beta", 0.0, 0.9)
    k_cand = trial.suggest_int("k_candidates", 10, 40)
    f1s = []
    for q, truth in VAL_QUERIES:
        got = retrieve(q, alpha, beta, k_cand)["Section"].tolist()
        f1s.append(prf(got, truth, K_FINAL)[2])
    return float(np.mean(f1s))

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_params
print("Best parameters:", best)
print(f"Best validation F1@5: {study.best_value:.4f}")

In [ ]:
import matplotlib.pyplot as plt

vals = [t.value for t in study.trials]
plt.figure(figsize=(7, 3.6))
plt.plot(range(1, len(vals) + 1), vals, "o-", label="Trial F1@5")
plt.plot(range(1, len(vals) + 1), np.maximum.accumulate(vals), "r--", label="Best so far")
plt.xlabel("Trial"); plt.ylabel("Mean F1@5 (validation)"); plt.legend(); plt.grid(alpha=.3)
plt.title("Optuna optimisation trace"); plt.tight_layout(); plt.show()

In [ ]:
def ranked_to_df(order, k=K_FINAL):
    return chunk_df.iloc[to_sections(order, k)]

def only_bm25(q):       return ranked_to_df(first_stage(q)["lists"]["bm25"])
def only_bge(q):        return ranked_to_df(first_stage(q)["lists"]["semantic"])
def bge_bm25(q):
    L = first_stage(q)["lists"]; return ranked_to_df(rrf_fuse([L["semantic"], L["bm25"]]))
def plus_topic(q):
    L = first_stage(q)["lists"]; return ranked_to_df(rrf_fuse([L["semantic"], L["bm25"], L["topic"]]))
def plus_graph(q):
    L = first_stage(q)["lists"]; return ranked_to_df(rrf_fuse(L.values()))
def plus_rrf_optuna(q):
    return retrieve(q, best["alpha"], 0.0, best["k_candidates"], use_ce=False)
def full_system(q):
    return retrieve(q, best["alpha"], best["beta"], best["k_candidates"])

configs = [("BM25", only_bm25), ("BGE-M3", only_bge), ("BGE-M3 + BM25", bge_bm25),
           ("+ BERTopic", plus_topic), ("+ Knowledge graph", plus_graph),
           ("+ RRF + Optuna", plus_rrf_optuna), ("Full system + Cross-Encoder", full_system)]

ablation = []
for name, fn in configs:
    s = summarise(evaluate(TEST_QUERIES, fn))
    ablation.append({"Configuration": name, **{k: round(v, 4) for k, v in s.items() if k != "Retrieval Accuracy@5"}})
ablation_df = pd.DataFrame(ablation)
ablation_df

In [ ]:
test_df = evaluate(TEST_QUERIES, full_system)
final_summary = summarise(test_df)

print("=" * 60)
print("FINAL RESULTS — 100-query held-out test set")
print("=" * 60)
print(f"alpha = {best['alpha']:.4f} (keyword weight {1 - best['alpha']:.4f}) | beta = {best['beta']:.4f} | "
      f"k_candidates = {best['k_candidates']} | k_final = {K_FINAL}")
for k, v in final_summary.items():
    print(f"{k:<22}: {v:.4f}")
print(f"Queries with a relevant section in top 5: {int(test_df['Hit@5'].sum())} / {len(test_df)}")

test_df.to_csv("Hybrid_RAG_Test_Results_100_queries.csv", index=False)
test_df.round(4)

In [ ]:
# Error analysis: what was missed / extra for the queries with no hit in the top 5
for _, r in test_df[test_df["Hit@5"] == 0].iterrows():
    truth, got = set(r["Ground Truth"].split(", ")), set(r["Retrieved"].split(", "))
    print(f"{r['Query']}\n  missed: {sorted(truth - got)}\n  retrieved: {sorted(got)}\n")

In [ ]:
from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])

PROMPT = """You are an expert Indian Legal Assistant.

Use ONLY the legal context provided below. Do not use outside knowledge.
If the answer is not present in the context, reply exactly:
"I couldn't find sufficient information in the provided legal documents."

<context>
{context}
</context>

<question>
{query}
</question>

Return the answer in exactly this format:

Relevant Section:
Explanation:
Penalty:
Connected Sections:
Summary:
"""

def build_context(res):
    return "\n".join(
        f'<section id="{r.Section}" chunk="{r.Chunk_ID}">\n{r.Chunk}\n</section>' for r in res.itertuples()
    )

def generate_answer(query, res, retries=3):
    msg = PROMPT.format(context=build_context(res), query=query)
    for attempt in range(retries):
        try:
            out = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[{"role": "user", "content": msg}],
                temperature=0.2,
            )
            return out.choices[0].message.content
        except Exception as e:                      # rate limit / transient error
            time.sleep(5 * (attempt + 1))
    return ""

def legal_assistant(query):
    res = retrieve(query, best["alpha"], best["beta"], best["k_candidates"])
    return generate_answer(query, res), res

answer, evidence = legal_assistant("Someone took my phone from my pocket without my knowledge. Which section applies?")
print(answer)

In [ ]:
# Answer-Level Accuracy: continuous overlap in [0, 1] between the section(s) named under
# "Relevant Section:" in the generated answer and the ground-truth sections.
RUN_ANSWER_EVAL = True      # 100 Groq calls; set False to skip

def named_sections(answer):
    m = re.search(r"Relevant Section:?(.*?)(?:Explanation:|$)", answer, flags=re.S | re.I)
    block = m.group(1) if m else answer
    return {"IPC_" + n.upper() for n in re.findall(r"\b(\d{1,3}[A-Za-z]?)\b", block)}

def answer_match(answer, truth):
    named, truth = named_sections(answer), set(truth)
    if not named or not truth:
        return 0.0
    return len(named & truth) / min(len(named), len(truth))      # overlap coefficient

if RUN_ANSWER_EVAL:
    scores = []
    for q, truth in TEST_QUERIES:
        ans, _ = legal_assistant(q)
        scores.append(answer_match(ans, truth))
        time.sleep(1)                                           # stay under Groq rate limits
    print(f"Answer-Level Accuracy (100 queries): {np.mean(scores):.4f}")

In [ ]:
import gradio as gr

def ui_answer(question):
    if not question.strip():
        return "Please enter a question.", ""
    answer, res = legal_assistant(question)
    evidence = "\n\n".join(f"**{r.Section}** (score {r.FinalScore:.3f})\n{r.Chunk}" for r in res.itertuples())
    return answer, evidence

demo = gr.Interface(
    fn=ui_answer,
    inputs=gr.Textbox(lines=2, label="Ask about the Indian Penal Code",
                      placeholder="e.g. What is the punishment for cheating?"),
    outputs=[gr.Textbox(label="Structured answer", lines=14), gr.Markdown(label="Retrieved evidence")],
    title="Hybrid RAG — Indian Statutory Information",
    description="BGE-M3 + BM25 + BERTopic + Knowledge Graph → RRF → Cross-Encoder → gpt-oss-20b",
    flagging_mode="never",
)
demo.launch(share=True)